# Semantic Product Search

Search the included catalog with English BGE embeddings and an in-memory LangChain vector store. This workflow returns product records; it does not generate answers with an LLM. Run the notebook from the project folder.


In [ ]:
from search import (CATALOG_PATH, EMBEDDING_MODEL, load_catalog_documents,
                    create_embeddings, build_search_index, find_products)
import pandas as pd


In [ ]:
df = pd.read_csv(CATALOG_PATH)
display(df[["name", "description"]].head())
print("Catalog rows:", len(df))


In [ ]:
documents = load_catalog_documents()
print("Indexed fields: product name and complete description")
print("Documents:", len(documents))


## Embedding model

`BAAI/bge-small-en-v1.5` is an English dense embedding model with 384 dimensions and a 512-token sequence limit. It is distinct from BGE-M3; this notebook does not implement BGE-M3 sparse or multi-vector retrieval. The query encoder uses the retrieval instruction recommended by the [model card](https://huggingface.co/BAAI/bge-small-en-v1.5). Longer descriptions may be truncated; there is no chunking step here.


In [ ]:
embeddings = create_embeddings()
query_vector = embeddings.embed_query("shirts with sun protection")
print("Embedding dimensions:", len(query_vector))


In [ ]:
store = build_search_index(documents, embeddings)


In [ ]:
query = "comfortable sheet sets"
matches = find_products(store, query, k=4, min_score=0.5)
if not matches:
    print("No products met the selected similarity filter.")
for i, (document, score) in enumerate(matches, 1):
    print(f"\n--- Match {i}: {document.metadata['name']} (similarity {score:.3f}) ---")
    print(document.metadata["description"])


## Explore another query

Change `query` above and rerun the search cell. The score filter is adjustable and has not been calibrated against relevance labels. Similarity scores are not probabilities. Both notebooks use downloaded pretrained models; no model training or retrieval benchmark is performed.
